# Терминал

Коваль:
https://www.youtube.com/watch?v=ys6VjUnWQCk
https://www.youtube.com/watch?v=U6CkCnJYhcU
https://www.youtube.com/watch?v=idUkymOQYJs
https://www.youtube.com/watch?v=OFhwMqXeQUE

Ахмедова:
https://www.youtube.com/watch?v=WC8MGi_sItE
https://www.youtube.com/watch?v=v1xb76zn2WQ

АРИАНА ЛОЛАЕВА:
https://www.youtube.com/watch?v=fdx7m7tgXdQ

Аранова Елизавета Варвара
https://www.youtube.com/watch?v=wAy_XfHCfuA

Слава Комиссаренко
https://www.youtube.com/watch?v=ExrG6AhR-wI
https://www.youtube.com/watch?v=k8Mixx5Q4vc

Алексей Квашонкин
https://www.youtube.com/watch?v=7GAr9pJeIp0

In [16]:
from pathlib import Path

url = "https://www.youtube.com/watch?v=OFhwMqXeQUE"


audio_path = "../data/!Notebook_test/audio/Idrak_first2.wav"
audio_name = Path(audio_path).stem

transcribe_path = "../data/!Notebook_test/transcripts_with_timestamp/"

print(transcribe_path + audio_name + ".json")

../data/!Notebook_test/transcripts_with_timestamp/Idrak_first2.json


# Тестирование отдельных функций

## Транскрибация локального аудиофайла в переменную (словарь)

In [9]:
import mlx_whisper

result = mlx_whisper.transcribe(
    str(audio_path),
    path_or_hf_repo="mlx-community/whisper-large-v3-turbo",
)

In [3]:
result

{'text': ' Какие у вас политические взгляды? Вот всегда-всегда вот такой нервный смешок. Но я вас понимаю, я тоже не распространяюсь о своих политических взглядах, но из-за того, что я немного медийный, обо мне иногда пишут, я про себя прочитал вот такое вот. Левацкое отродье. Вот так написали. И мне не понравилось, потому что я сам не уверен, кто я, решил загуглить, как понять, какие у тебя политические взгляды. И мне попалась там диаграмма политических взглядов. Вам попадалось такое? Диаграмма, по которой ты определяешь, кто ты. И оказалось, что я ультраправый радикал вообще. По многим взглядам совпало, например, отношение к гомосексуализму. У меня такое уже крайне негативное. Взять, если там лесбиянок, например. Я вообще не пойму, как можно хотеть женщин. Для меня это загадка. Или, например, отношение к эмигрантам. Я вот сам, как только, я три года назад переехал в Европу. Я как только переехал, я подумал, закрывайте ворота, нам эти обезьяны тут не нужны. Мы не для того, да, приехал

## Фильтрация словаря,получение только нужных колонок

In [10]:
filtered_result = {
    "text": result["text"],
    "segments": [
        {
            "id": segment["id"],
            "start": segment["start"],
            "end": segment["end"],
            "text": segment["text"],
        }
        for segment in result.get("segments", [])
    ],
}

In [11]:
filtered_result

{'text': ' Какие у вас политические взгляды? Вот всегда-всегда вот такой нервный смешок. Но я вас понимаю, я тоже не распространяюсь о своих политических взглядах, но из-за того, что я немного медийный, обо мне иногда пишут, я про себя прочитал вот такое вот. Левацкое отродье. Вот так вот. И мне не понравилось, потому что я сам не уверен, кто я, решил загуглить, как понять, какие у тебя политические взгляды. И мне попалась там диаграмма политических взглядов. Вам попадалось такое? Диаграмма, по которой ты определяешь, кто ты. И оказалось, что я ультраправый радикал вообще. По многим взглядам совпало, например, отношение к гомосексуализму. У меня такое уже крайне негативное. Взять, если там лесбиянок, например. Я вообще не пойму, как можно хотеть женщин. Для меня это загадка. Или, например, отношение к эмигрантам. Я вот сам, как только, я три года назад переехал в Европу. Я как только переехал, я подумал, закрывайте ворота, нам эти обезьяны тут не нужны. Мы не для того, да, приехали в Е

## Запись после фильтрация в json файл

In [17]:
import json

with open(transcribe_path + audio_name + ".json", "w", encoding="utf-8") as f:
    json.dump(filtered_result, f, ensure_ascii=False, indent=4)

## Чтение json файла в pandas DF

In [36]:
import json
import pandas as pd

# Загрузка JSON из файла или строки
with open(
    "../data/transcripts_with_timestamp/Node quietly dropped its biggest update in years....json",
    "r",
    encoding="utf-8",
) as f:
    data = json.load(f)

# Преобразуем ключ 'segments' в DataFrame
df = pd.DataFrame(data["segments"])

# Проверка результата
df.head()


,id,start,end,text
0,0,0.28,3.88,Node.js just did something huge and almost no...
1,1,3.88,8.46,"For the past decade, TypeScript has been repl..."
2,2,8.46,10.28,to back-end APIs.
3,3,10.28,13.24,"Everyone writes TypeScript now, whether they ..."
4,4,13.24,16.40,"And naturally, modern JavaScript runtimes had..."


## Работа с LLM

In [9]:
import os
from google import genai
import json


# Загрузка JSON из файла или строки
with open(
    "../data/transcripts_with_timestamp/Идрак Мирзализаде. Стендап концерт Это не я придумал.json",
    "r",
    encoding="utf-8",
) as f:
    data = json.load(f)

data = data["segments"]

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

prompt = f"""
Вы — эксперт по тематическому анализу аудио/видео транскриптов.

На входе вы получаете список объектов вида:

[
{{'id': 0,
  'start': 0.28,
  'end': 3.88,
  'text': 'Какие у вас политические взгляды?'}},
 {{'id': 1,
  'start': 3.88,
  'end': 8.46,
  'text': 'Вот всегда, всегда вот такой нервный смешок.'}}
]

Ваша задача:
1. Проанализировать все фрагменты текста и выделить глобальные ключевые темы (например, "политические взгляды", "медиа", "самоидентификация").
2. Для каждой темы определить временной диапазон: от минимальной до максимальной отметки `timestamp`, относящейся к сообщениям этой темы.
3. Вернуть результат **строго в формате JSON**, без каких-либо комментариев, префиксов, пояснений или форматирования.

Формат ожидаемого результата:
{{
"политические взгляды": {{"start_sec": 1.564, "end_sec": 3.530}},
"медиа": {{"start_sec": 8.951, "end_sec": 9.312}},
"самоидентификация": {{"start_sec": 16.077, "end_sec": 17.647}}
}}

Вот расшифровка:
---
{data}
---
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config={
        "response_mime_type": "application/json",
    },
)

response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        role='model'
      ),
      finish_reason=<FinishReason.MAX_TOKENS: 'MAX_TOKENS'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  sdk_http_response=HttpResponse(
    headers=<dict len=11>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    prompt_token_count=45683,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=45683
      ),
    ],
    thoughts_token_count=65535,
    total_token_count=111218
  )
)

In [3]:
formatted_text = json.loads(response.text)

formatted_text

{'Политические взгляды и Самоидентификация': {'start_sec': 0.0,
  'end_sec': 112.32},
 'Национализм и Стереотипы': {'start_sec': 112.32, 'end_sec': 264.96},
 'Расизм и Толерантность': {'start_sec': 264.96, 'end_sec': 488.68},
 'Свобода слова и Культура отмены': {'start_sec': 488.68, 'end_sec': 599.48},
 'Проблема тупости в обществе': {'start_sec': 598.48, 'end_sec': 967.48},
 'Отношения и Общественные Проблемы': {'start_sec': 967.48, 'end_sec': 1906.2},
 'Мужская анатомия и Мужские Проблемы': {'start_sec': 1908.32,
  'end_sec': 2939.32},
 'Вредные Привычки и Осознанность': {'start_sec': 2942.02, 'end_sec': 3032.8},
 'География, Жизнь за границей и Национальность': {'start_sec': 3033.26,
  'end_sec': 3105.8599999999997},
 'Такси, Стереотипы и Религия': {'start_sec': 3107.74, 'end_sec': 3152.72},
 'Комедия, Искусство и Взаимодействие с аудиторией': {'start_sec': 3160.02,
  'end_sec': 3268.12}}